In [47]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from func.UsefulFunctions import prettifyGenres

This jupyter notebook inspects the contents of the artists (that is individual performers/creatives) to gain an understanding of their statistics. The sou

In [48]:

artistDF = pl.read_parquet("/home/aja25/Documents/fromCluster/personMusicBrainzData.parquet")
#De-duplicate the data
artistDF = artistDF.unique()

#How many unique artists are there now. This data is now based off of the number of unique

num_of_ids = artistDF.select(
    pl.col("id")
).unique().count().item(0,0)
print(f"There are {num_of_ids} unique IDs in the dataset (that is a snapshot from 18/07/2026).")

There are 1574608 unique IDs in the dataset (that is a snapshot from 18/07/2026).


In [49]:
#How many people do not have a name in this dataset?
num_people_with_no_names = artistDF.select(
    pl.col("name"),
    pl.col("id")
).unique().null_count()
print(num_people_with_no_names)
print(f"There are {num_people_with_no_names.item(0,0)} entities with no name, and {num_people_with_no_names.item(0,1)} entities with no ids.")

shape: (1, 2)
┌──────┬─────┐
│ name ┆ id  │
│ ---  ┆ --- │
│ u32  ┆ u32 │
╞══════╪═════╡
│ 0    ┆ 0   │
└──────┴─────┘
There are 0 entities with no name, and 0 entities with no ids.


In [50]:
#check if there are any duplicates in ids
ids = artistDF.select(
    pl.col("id")
).is_duplicated()
print(ids.value_counts())

print("Based off of the lack of trues in the is_duplicated function. All rows are unique after the deduplication ")

shape: (1, 2)
┌───────┬─────────┐
│       ┆ count   │
│ ---   ┆ ---     │
│ bool  ┆ u32     │
╞═══════╪═════════╡
│ false ┆ 1574608 │
└───────┴─────────┘
Based off of the lack of trues in the is_duplicated function. All rows are unique after the deduplication 


In [51]:
#Do a count for the countries for the dataset.
countryCount = artistDF.select(pl.col("country").value_counts(sort=True)).unnest("country")
print(countryCount)

shape: (252, 2)
┌─────────┬────────┐
│ country ┆ count  │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ null    ┆ 655691 │
│ US      ┆ 220745 │
│ GB      ┆ 81004  │
│ JP      ┆ 75389  │
│ DE      ┆ 71965  │
│ …       ┆ …      │
│ HM      ┆ 1      │
│ CX      ┆ 1      │
│ PN      ┆ 1      │
│ SH      ┆ 1      │
│ NU      ┆ 1      │
└─────────┴────────┘


The most common country for individual performers (outside of null which trumps it) is US followed by GB and Japan.

In [52]:
#What is the gender split like for the entities
genderCount = artistDF.select(pl.col("gender").value_counts(sort=True)).unnest("gender")
genderCountPercent = artistDF.select(pl.col("gender").value_counts(sort=True, normalize=True, name="Fraction")).unnest("gender")
print(genderCount)
genderChart = (
    alt.Chart(genderCount).mark_bar().encode(
    y="count",
    x="gender",
    )
    .properties(width=500, title="Gender Distribution of Music Brainz")
)
genderChart.encoding.x.title = "Genders"
genderChart.encoding.y.title = "Counts"
genderChart.show()

shape: (6, 2)
┌────────────────┬────────┐
│ gender         ┆ count  │
│ ---            ┆ ---    │
│ str            ┆ u32    │
╞════════════════╪════════╡
│ Male           ┆ 947583 │
│ null           ┆ 345419 │
│ Female         ┆ 277599 │
│ Non-binary     ┆ 2344   │
│ Other          ┆ 1244   │
│ Not applicable ┆ 419    │
└────────────────┴────────┘


alt.Chart(...)

In [53]:
print(genderCountPercent)

shape: (6, 2)
┌────────────────┬──────────┐
│ gender         ┆ Fraction │
│ ---            ┆ ---      │
│ str            ┆ f64      │
╞════════════════╪══════════╡
│ Male           ┆ 0.60179  │
│ null           ┆ 0.219368 │
│ Female         ┆ 0.176297 │
│ Non-binary     ┆ 0.001489 │
│ Other          ┆ 0.00079  │
│ Not applicable ┆ 0.000266 │
└────────────────┴──────────┘


Overwhelmingly, the dataset skews towards men with ~60.2% of the dataset being men. More individuals do not have their gender attributed to them than female individuals.

In [54]:
artistDFPandas = artistDF.to_pandas(use_pyarrow_extension_array=True)
artistDFPandas['genreList'] = artistDFPandas['genres'].map(prettifyGenres)
display(artistDFPandas.head(5))
display(artistDFPandas[artistDFPandas.name == "Madonna"])
artistDF = pl.from_pandas(artistDFPandas)
madonna = artistDF.filter(pl.col("name") == "Madonna")
display(madonna.head(5))


,id,name,type,gender,country,genres,isnis,ipis,life-span,genreList
0,dbef111e-c888-4ee4-a3b4-42a5a10a5a98,The IZA,Person,<NA>,<NA>,[],['0000000372628166'],[],"{'end': None, 'begin': None, 'ended': False}",[]
1,3f6b422b-e92b-4a2a-9f4a-9ba36f9dc21a,tekcoh_top,Person,Male,<NA>,[],[],[],"{'end': None, 'begin': None, 'ended': False}",[]
2,aa94ba3e-f11f-46a3-a743-a25625d2361a,加藤久美子,Person,Female,JP,[],[],[],"{'end': None, 'begin': None, 'ended': False}",[]
3,e8b2932e-c33e-43c6-afb1-caf6c967629e,Giovanni Inghilleri,Person,Male,IT,[],['0000000382898426'],[],"{'end': '1959-12-09', 'begin': '1894-03-09', '...",[]
4,7ac47fa9-51fa-4450-b5b5-87dff30d8131,Rayan Myers,Person,Male,RU,[],[],[],"{'end': None, 'begin': None, 'ended': False}",[]


,id,name,type,gender,country,genres,isnis,ipis,life-span,genreList
102112,3fd13140-6857-489a-9e53-a4e112b8354b,Madonna,Person,Female,LB,[],[],[],"{'end': None, 'begin': None, 'ended': False}",[]
1502594,79239441-bfd5-4981-a70c-55c3f15c1287,Madonna,Person,Female,US,[{'id': '930ef127-3653-4677-9b95-ecc90c7c1a14'...,['000000010782262X'],['00125053706' '00128246872' '00211854787'],"{'end': None, 'begin': '1958-08-16', 'ended': ...","[art pop, bubblegum pop, club, contemporary r&..."


id,name,type,gender,country,genres,isnis,ipis,life-span,genreList
str,str,str,str,str,list[struct[4]],list[str],list[str],struct[3],list[str]
"""3fd13140-6857-489a-9e53-a4e112…","""Madonna""","""Person""","""Female""","""LB""",[],[],[],"{null,null,false}",[]
"""79239441-bfd5-4981-a70c-55c3f1…","""Madonna""","""Person""","""Female""","""US""","[{""930ef127-3653-4677-9b95-ecc90c7c1a14"",7,"""",""art pop""}, {""6fd83b83-9b63-47fa-a208-5f9e762f40ab"",1,"""",""bubblegum pop""}, … {""45eb1d9c-588c-4dc8-9394-a14b7c8f02bc"",1,"""",""trip hop""}]","[""000000010782262X""]","[""00125053706"", ""00128246872"", ""00211854787""]","{null,""1958-08-16"",false}","[""art pop"", ""bubblegum pop"", … ""trip hop""]"


In [55]:
#Get the most common genre

genreCount = artistDF.select(
    pl.col("id"),
    pl.col("name"),
    pl.col("gender"),
    pl.col("genreList").alias("Genre"),
)

genreCount = genreCount.explode("Genre", empty_as_null=True, keep_nulls=True)
genreCount = genreCount.unique()
number_of_genres = genreCount.select(
    pl.col("Genre").value_counts(sort=True)
).unnest("Genre")
print(number_of_genres)


shape: (1_561, 2)
┌──────────────────┬─────────┐
│ Genre            ┆ count   │
│ ---              ┆ ---     │
│ str              ┆ u32     │
╞══════════════════╪═════════╡
│ null             ┆ 1481063 │
│ hip hop          ┆ 16678   │
│ jazz             ┆ 12343   │
│ pop              ┆ 8408    │
│ electronic       ┆ 6811    │
│ …                ┆ …       │
│ campursari       ┆ 1       │
│ forró eletrônico ┆ 1       │
│ autonomic        ┆ 1       │
│ wassoulou        ┆ 1       │
│ abhang           ┆ 1       │
└──────────────────┴─────────┘


Most individuals do not have a genre attached to them based off of MusicBrainz's list of genres but the most popular genre is hip hop followed by Jazz music.

In [56]:
#Look for the gender and genre split using group by.
genreCount.group_by(["gender",'Genre']).agg(pl.col('id').count())

gender,Genre,id
str,str,u32
"""Male""","""ars antiqua""",2
null,"""vaporwave""",78
"""Female""","""christian rock""",8
"""Male""","""c-pop""",34
"""Male""","""detroit techno""",32
…,…,…
"""Female""","""carimbó""",2
null,"""art pop""",10
null,"""spoken word""",15


In [57]:
#Most common gender-country pair.
artistDF.group_by(["gender","country"]).agg(pl.col('id').count())

gender,country,id
str,str,u32
null,"""QA""",1
"""Male""","""MA""",573
null,"""VG""",3
"""Non-binary""","""AT""",5
"""Female""","""GD""",10
…,…,…
null,"""HR""",127
"""Female""","""TO""",1
"""Male""","""AE""",158
